`pip install tensorflow`

In [ ]:
# ==============================================================================
# KROK 5: GWARANTOWANY PODRĘCZNIKOWY OVERFITTING (Mało danych + duża sieć)
# ==============================================================================
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

# 1. Generujemy MAŁY zbiór ze SPOKĄ ILOŚCIĄ SZUMU (idealne warunki do przeuczenia)
X_small, y_small = make_moons(n_samples=100, noise=0.35, random_state=42)

# Zbiór treningowy ma tylko 60 próbek!
X_train_nn, X_val_nn, y_train_nn, y_val_nn = train_test_split(
    X_small, y_small, test_size=0.4, random_state=42
)

# 2. Budujemy potężną sieć (tysiące parametrów na zaledwie 60 wierszy danych!)
tf.keras.backend.clear_session()
tf.random.set_seed(42)

model_overfit = Sequential([
    Dense(128, activation='relu', input_shape=(2,)),
    Dense(64, activation='relu'),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])

model_overfit.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.005), # Nieco wyższy LR dla szybszego efektu
    loss='binary_crossentropy', 
    metrics=['accuracy']
)

# Trenujemy przez 150 epok
history_overfit = model_overfit.fit(
    X_train_nn, y_train_nn,
    validation_data=(X_val_nn, y_val_nn),
    epochs=150,
    verbose=0
)

# 3. Rysujemy wykres
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
axes[0].plot(history_overfit.history['loss'], label='Treningowa strata (loss)', lw=2)
axes[0].plot(history_overfit.history['val_loss'], label='Walidacyjna strata (val_loss)', color='red', lw=2)
axes[0].set_title("Ewidentny Overfitting na Funkcji Straty (Loss)\nCzerwona linia ucieka drastycznie w górę!", fontsize=12)
axes[0].set_xlabel("Epoka")
axes[0].set_ylabel("Loss")
axes[0].legend()

# Accuracy
axes[1].plot(history_overfit.history['accuracy'], label='Treningowa celność', lw=2)
axes[1].plot(history_overfit.history['val_accuracy'], label='Walidacyjna celność', color='red', lw=2)
axes[1].set_title("Celność: 100% na treningu vs Spadek na walidacji", fontsize=12)
axes[1].set_xlabel("Epoka")
axes[1].set_ylabel("Accuracy")
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"Końcowy stan w 150. epoce:")
print(f" • Strata treningowa:   {history_overfit.history['loss'][-1]:.4f}  (Prawie idealne zero!)")
print(f" • Strata walidacyjna:  {history_overfit.history['val_loss'][-1]:.4f}  (EKSPLOZJA BŁĘDU!)")

In [ ]:
# ==============================================================================
# KROK 6: Wdrożenie Dropout + EarlyStopping na TYM SAMYM trudnym zbiorze
# ==============================================================================
from tensorflow.keras.layers import Dropout
from tensorflow.keras.callbacks import EarlyStopping

tf.keras.backend.clear_session()
tf.random.set_seed(42)

# Mniejsza architektura (Brzytwa Ockhama) + mocny Dropout
model_tuned = Sequential([
    Dense(32, activation='relu', input_shape=(2,)),
    Dropout(0.4),                                   # Losowe wyłączanie 40% neuronów
    Dense(16, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

model_tuned.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.005),
    loss='binary_crossentropy', 
    metrics=['accuracy']
)

# Zatrzymujemy trening, gdy val_loss przestanie spadać
early_stop = EarlyStopping(
    monitor='val_loss', 
    patience=15, 
    restore_best_weights=True
)

history_tuned = model_tuned.fit(
    X_train_nn, y_train_nn,
    validation_data=(X_val_nn, y_val_nn),
    epochs=150,
    callbacks=[early_stop],
    verbose=0
)

# Wykres po naprawie
plt.figure(figsize=(10, 5))
plt.plot(history_tuned.history['loss'], label='Treningowa strata (loss)', lw=2)
plt.plot(history_tuned.history['val_loss'], label='Walidacyjna strata (val_loss)', color='green', lw=2)
plt.title(f"Poskromienie przeuczenia!\nTrening zatrzymany w {len(history_tuned.history['loss'])}. epoce (Early Stopping)", fontsize=13)
plt.xlabel("Epoka")
plt.ylabel("Loss")
plt.legend()
plt.show()